In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import joblib
import os

BASE = r'C:\Users\Anushka Thite\Desktop\smart-loan-system'
RAW_PATH = os.path.join(BASE, 'data', 'raw', 'Loan Approval Prediction Dataset.csv')
PROCESSED_DIR = os.path.join(BASE, 'data', 'processed')
MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print("Original shape:", df.shape)

drop_cols = ['Application_ID', 'Monthly_Income', 'Bank_Balance', 'Collateral_Value']
df = df.drop(columns=drop_cols)
print("After drop shape:", df.shape)

mask_no_collateral = df['Collateral'] == 'No'
df.loc[mask_no_collateral, 'Property_Value'] = 0
df.loc[mask_no_collateral, 'Loan_to_Value'] = 0
for col in ['Property_Value', 'Loan_to_Value']:
    median_yes = df.loc[df['Collateral'] == 'Yes', col].median()
    df[col] = df[col].fillna(median_yes)

df['EMI_to_Income']   = df['Monthly_EMI'] / (df['Annual_Income'] / 12 + 1)
df['Loan_to_Income']  = df['Loan_Amount'] / (df['Annual_Income'] + 1)
df['Savings_to_Loan'] = df['Savings'] / (df['Loan_Amount'] + 1)
df['Net_Worth']       = df['Savings'] + df['Investments'] - df['Existing_Loan_Amount']
df['Has_Collateral']  = (df['Collateral'] == 'Yes').astype(int)
print("After FE shape:", df.shape)

df['Loan_Status'] = df['Loan_Status'].map({'Approved': 1, 'Rejected': 0})
print("\nTarget distribution:")
print(df['Loan_Status'].value_counts())

X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print(f"\nAfter preprocessing: Train {X_train_proc.shape}, Test {X_test_proc.shape}")

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)
print(f"After SMOTE: {X_train_res.shape}")
print(f"Balanced target: {np.bincount(y_train_res)}")

np.save(os.path.join(PROCESSED_DIR, 'X_train.npy'), X_train_res)
np.save(os.path.join(PROCESSED_DIR, 'X_test.npy'), X_test_proc)
np.save(os.path.join(PROCESSED_DIR, 'y_train.npy'), y_train_res)
np.save(os.path.join(PROCESSED_DIR, 'y_test.npy'), y_test.values)
joblib.dump(preprocessor, os.path.join(MODELS_DIR, 'preprocessor.pkl'))

feature_names = (numeric_cols +
                 list(preprocessor.named_transformers_['cat']
                      .named_steps['encoder']
                      .get_feature_names_out(categorical_cols)))
joblib.dump(feature_names, os.path.join(MODELS_DIR, 'feature_names.pkl'))

print("\n✅ Preprocessing complete.")

Original shape: (20000, 39)
After drop shape: (20000, 35)
After FE shape: (20000, 40)

Target distribution:
Loan_Status
1    12856
0     7144
Name: count, dtype: int64

Train: (16000, 39), Test: (4000, 39)

After preprocessing: Train (16000, 63), Test (4000, 63)
After SMOTE: (20570, 63)
Balanced target: [10285 10285]

✅ Preprocessing complete.
